# Documentary Transcript Theme Analysis — Turn-Merge-Coarse Chunking (Approach #4)
**Pipeline:** WhisperX `.txt` transcripts → coarse-category-merged turn chunks → BERTopic → Visualizations

This notebook implements **Approach #4 (Turn-merge-coarse, speaker-structure)** from the 10-approaches design: a coherence-vs-chunk-size compromise on top of Approach #3. Chunking is still turn-based and speaker-structure-driven — no clock involved — but the merge boundary now checks the *coarse* category instead of the exact fine-grained speaker label. Adjacent turns from different specific speakers merge into one chunk as long as they share the same coarse category (e.g. `EYEWITNESS_01` → `EYEWITNESS_02` back-to-back becomes one chunk, where Approach #3 would have split them into two). This reduces over-fragmentation from #3's many short, single-speaker chunks, while still respecting category boundaries — a chunk never blends two *different* coarse categories together.

Upload your `.txt` transcript files to the Colab session (Files panel on the left) before running.

## 1. Install Dependencies

## 2. Imports

In [1]:
import os
import re
import glob
from pathlib import Path

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
import plotly.io as pio

pio.renderers.default = 'notebook'
print('All imports OK')

All imports OK


## 3. Load & Parse Transcripts

WhisperX `.txt` format: `[MM:SS] SPEAKER_XX: text`

**Chunking strategy:** turn-merge-coarse, not time-window and not turn-strict. Each chunk is one contiguous run of lines that share the same *coarse* speaker category — the chunk stays open across a fine-grained speaker change as long as the coarse category doesn't change (e.g. `EYEWITNESS_01` → `EYEWITNESS_02` back-to-back merges into one chunk), and only closes once the coarse category itself changes (e.g. `EYEWITNESS` → `PROFESSIONAL`). There is still no clock involved: a chunk's length is however long that category keeps talking, across however many individual speakers contribute to it.

This means fewer, longer chunks than Approach #3's exact-speaker-boundary chunking, while `dominant_category` stays exact (every line in a chunk shares the same coarse category by construction) — the difference from #3 is only in *how many individual speakers* a chunk can span, not in category certainty.

**Tuning tips:**
- `MIN_WORDS`: kept at the same floor as #3 — chunks are still turn-based rather than clock-based, just merged across more turns, so very short isolated runs can still occur and get dropped here
- Expect a chunk count between #1/#2's (coarsest) and #3's (most fragmented) — check the diagnostics printed by the cell below and retune `min_cluster_size` against your actual count rather than assuming #3's value transfers directly
- If one coarse category's run still ends up very long (the same long-monologue case flagged in #3, just now possibly spanning several speakers of that category), that whole run still becomes one chunk here — Approach #5 (hybrid-capped) is what re-splits oversized chunks; this notebook intentionally leaves them whole to isolate the effect of coarse-category merging on its own

In [3]:
import os
import re
import glob
from pathlib import Path
from docx import Document
from collections import Counter

TIMESTAMP_RE = re.compile(
    r'^(?:source:\s∗\d+source:\s*\d+\s*)?'
    r'(?:\[)?'
    r'(?P<start>\d{1,2}:\d{2}(?::\d{2})?)'
    r'(?:\])?'
    r'[:\s]*'
    r'(?:(?:SPEAKER_\w+|UNKNOWN_SPEAKER)[\s:]+)?'
    r'(?P<text>.+)$'
)

SPEAKER_RE = re.compile(
    r'^(?P<speaker>(?:UNKNOWN|[A-Z][A-Z0-9_\s\.\'\-]{1,40}))(?:[:\s]+)(?P<text>.+)$|'
    r'^(?P<speaker2>[A-Z][a-z]+(?:\s[A-Z][a-z]+){0,3}):(?P<text2>.+)$'
)

def timestamp_to_seconds(ts: str) -> float:
    parts = [float(p) for p in ts.strip().split(':')]
    if len(parts) == 2: return parts[0] * 60 + parts[1]
    if len(parts) == 3: return parts[0] * 3600 + parts[1] * 60 + parts[2]
    return 0.0


# Match timestamp, optional speaker label (followed by colon), and dialogue text
LINE_RE = re.compile(
    r'^(?:source:\s*\d+\s*)?'          # Optional source tag
    r'(?:\[)?(?P<start>\d{1,2}:\d{2}(?::\d{2})?)?'  # Timestamp [MM:SS] or [HH:MM:SS]
    r'[\s:]*'                          # Optional spacing
    r'(?:(?P<speaker>[A-Za-z0-9_\- ]+):)?'  # Speaker label MUST be followed by a colon
    r'[\s]*'                           # Spacing after colon
    r'(?P<text>.+)$'                   # Dialogue text
)

def clean_and_parse_line(line: str) -> dict | None:
    line = line.strip()
    if not line: 
        return None
    
    m = LINE_RE.search(line)
    if not m:
        return None
        
    text_content = m.group('text').strip()
    normalized = text_content.upper()
    noise = ['[SIL.]', '[MUSIC]', 'BEGIN TRANSCRIPT:', 'END TRANSCRIPT', 'TRANSCRIPT OF VIDEO FILE:']
    if normalized in noise or normalized.startswith('___'):
        return None

    speaker = m.group('speaker')
    if speaker:
        speaker = speaker.strip().upper()
    else:
        speaker = 'NARRATOR'

    # Filter out empty or ultra-short noise segments
    if len(text_content.split()) < 2:
        return None

    return {
        'seconds': timestamp_to_seconds(m.group('start')),
        'speaker': speaker,
        'text': text_content
    }

def parse_transcript(path: str) -> list[dict]:
    segments = []
    ext = Path(path).suffix.lower()
    if ext == '.txt':
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                seg = clean_and_parse_line(line)
                if seg: segments.append(seg)
    elif ext == '.docx':
        doc = Document(path)
        paras = [p.text.strip() for p in doc.paragraphs]
        i = 0
        while i < len(paras):
            para = paras[i]
            # check if this paragraph is a standalone timestamp
            ts_only = re.match(r'^\d{1,2}:\d{2}(?::\d{2})?$', para)
            if ts_only and i + 1 < len(paras):
                # combine timestamp with next paragraph and parse as one line
                combined = para + ' ' + paras[i + 1]
                seg = clean_and_parse_line(combined)
                if seg: segments.append(seg)
                i += 2
            else:
                # timestamp and text already on same line
                seg = clean_and_parse_line(para)
                if seg: segments.append(seg)
                i += 1
    return segments

CATEGORY_RE = re.compile(r'^(.*?)_\d+$')

def coarse_category(speaker: str) -> str:
    """Strip the trailing _NN person-index from a speaker label to get its category.
    e.g. 'BEREAVED_ADVOCATE_REFORM_05' -> 'BEREAVED_ADVOCATE_REFORM'.
    """
    m = CATEGORY_RE.match(speaker.strip().upper())
    return m.group(1) if m else speaker.strip().upper()

def chunk_by_turn_merge_coarse(segments: list[dict], min_words: int) -> list[dict]:
    """Turn-merge-coarse chunking (Approach #4).

    Same turn-based logic as Approach #3 -- no clock, chunk boundaries are
    driven entirely by speaker structure -- but the boundary condition checks
    the COARSE category instead of the exact fine-grained speaker label.
    Adjacent turns merge into one chunk as long as they share the same coarse
    category, even if the fine-grained speaker changes (e.g. EYEWITNESS_01 ->
    EYEWITNESS_02 back-to-back both land in the same chunk, since both
    collapse to coarse category EYEWITNESS). The chunk only closes when the
    COARSE category actually changes (e.g. EYEWITNESS -> PROFESSIONAL).

    Because a chunk can now span multiple distinct speakers, 'speaker' on the
    resulting chunk is the speaker of its first line (kept for display /
    provenance) rather than a single exact speaker -- only
    'dominant_category' is structurally guaranteed exact, since every line in
    the chunk shares it by construction.
    """
    if not segments: return []
    chunks, buffer = [], []
    current_category = coarse_category(segments[0]['speaker'])

    def close_chunk(buf):
        if not buf: return None
        text = ' '.join(seg['text'] for seg in buf)
        if len(text.split()) < min_words:
            return None
        speakers_in_chunk = sorted(set(seg['speaker'] for seg in buf))
        return {
            'text': text,
            'start_time': buf[0]['seconds'],
            'end_time': buf[-1]['seconds'],
            'speaker': buf[0]['speaker'],  # first speaker in the merged run
            'speakers': speakers_in_chunk,  # all distinct speakers merged into this chunk
            'dominant_category': coarse_category(buf[0]['speaker']),  # exact -- every line in buf shares this by construction
            'n_lines': len(buf),
        }

    for seg in segments:
        seg_category = coarse_category(seg['speaker'])
        if seg_category != current_category:
            chunk = close_chunk(buffer)
            if chunk: chunks.append(chunk)
            buffer = []
            current_category = seg_category
        buffer.append(seg)

    if buffer:
        chunk = close_chunk(buffer)
        if chunk: chunks.append(chunk)
    return chunks

In [4]:
# ── CONFIG ─────────────────────────────────────────────────────────────────
TXT_FOLDER  = './transcripts_labeled'
MIN_WORDS   = 15   # turns are naturally shorter than time windows; lower floor than #1/#2
OUTPUT_DIR  = './turn_merge_coarse_bertopic'
os.makedirs(OUTPUT_DIR, exist_ok=True)
# ───────────────────────────────────────────────────────────────────────────

# Grab both types of documents
file_paths = glob.glob(os.path.join(TXT_FOLDER, '*.txt')) + glob.glob(os.path.join(TXT_FOLDER, '*.docx'))
assert file_paths, f'No transcript files found in {TXT_FOLDER!r}'

all_chunks, all_doc_names, all_metadata = [], [], []

for path in sorted(file_paths):
    name = Path(path).name
    stem = Path(path).stem

    segs = parse_transcript(path)
    if not segs: continue

    chunks = chunk_by_turn_merge_coarse(segs, min_words=MIN_WORDS)

    all_chunks.extend([c['text'] for c in chunks])
    all_doc_names.extend([stem] * len(chunks))
    all_metadata.extend([
        {
            'doc': stem,
            'start': c['start_time'],
            'end': c['end_time'],
            'speaker': c['speaker'],
            'category': c['dominant_category'],
            'n_lines': c['n_lines'],
        }
        for c in chunks
    ])

    print(f'  {name}: {len(segs)} lines → {len(chunks)} coarse-merged turn-chunks')

print(f'\nTotal chunks loaded for BERTopic: {len(all_chunks)}')

word_counts = [len(t.split()) for t in all_chunks]
print(f'Chunk word count — min: {min(word_counts)}, median: {sorted(word_counts)[len(word_counts)//2]}, '
      f'max: {max(word_counts)}, mean: {sum(word_counts)/len(word_counts):.1f}')

line_counts = [m['n_lines'] for m in all_metadata]
print(f'Lines per chunk    — min: {min(line_counts)}, median: {sorted(line_counts)[len(line_counts)//2]}, '
      f'max: {max(line_counts)}')

   Chi-Town Guns, Gun Violence and The NRA.txt: 825 lines → 1 coarse-merged turn-chunks
  2nd Chance.txt: 1143 lines → 1 coarse-merged turn-chunks
  91%- A Film About Guns in America.txt: 163 lines → 1 coarse-merged turn-chunks
  All These Sons.txt: 1277 lines → 1 coarse-merged turn-chunks
  AlwaysInSeason.txt: 807 lines → 1 coarse-merged turn-chunks
  American Tragedy.txt: 166 lines → 1 coarse-merged turn-chunks
  Behind The Bullet.txt: 821 lines → 1 coarse-merged turn-chunks


AttributeError: 'NoneType' object has no attribute 'strip'

## 4. Build & Fit BERTopic Model

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

spoken_language_extras = [
    'like', 'know', 'just', 'don', 'yeah', 'okay', 'got', 'said',
    'going', 'right', 'think', 'want', 'really', 'thing', 'things',
    'say', 'way', 'time', 'come', 'came', 'went', 'get', 'go',
    'look', 'mean', 'actually', 'basically', 'literally', 'kind',
    'lot', 'little', 'big', 'good', 'great', 'need', 'tell',
    've', 'll', 're', 'd', 's'  # tokenization artifacts from contractions
]

personal_names = [
    'gabby', "gabrielle", 'mark', 'giffords', 'kelly', 'nathan', 'joaquin',
    'carrie', 'sarah', 'eddie', 'richard', 'aaron', 'carol',
    'claude', 'tom', 'joy', 'colin', 'goddard'
]

spoken_language_extras = spoken_language_extras + personal_names

custom_stopwords = list(stopwords.words('english')) + spoken_language_extras

# ── Embedding model (runs on T4 GPU automatically) ─────────────────────────
embedding_model = SentenceTransformer('all-mpnet-base-v2')

# ── UMAP: lower n_neighbors = finer local structure ────────────────────────
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    random_state=42
)

# ── HDBSCAN: min_cluster_size controls topic granularity ───────────────────
# NOTE: turn-merge-coarse chunking sits between #1/#2's 60s windows and #3's
# turn-strict chunks in granularity -- merging adjacent same-coarse-category
# turns reduces #3's chunk count (fewer, longer chunks) without going all the
# way back to a fixed clock. Started here at the same 20 used for #3 as a
# reasonable starting point, but check your actual chunk count after running
# the cell above and adjust: if it's much closer to #1/#2's chunk count,
# consider moving min_cluster_size back down toward their value of 10.
hdbscan_model = HDBSCAN(
    min_cluster_size=20,
    min_samples=5,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

vectorizer_model = CountVectorizer(
    stop_words=custom_stopwords,
    min_df=2,
    ngram_range=(1, 2)
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    top_n_words=10,
    verbose=True
)

topics, probs = topic_model.fit_transform(all_chunks)

print(f"Chunks available: {len(all_chunks)}")
print(f"Sample: {all_chunks[0][:80]}")
print(f"UMAP n_neighbors: {umap_model.n_neighbors}")
print(f"HDBSCAN min_cluster_size: {hdbscan_model.min_cluster_size}")

print(f'\nDiscovered {len(set(topics)) - 1} topics '
      f'(plus {topics.count(-1)} outlier chunks)')

## 4a. Speaker-Category Breakdown (Structural, Not Metadata)

Like Approach #3, `category` here is structural rather than a statistically *dominant* approximation across an arbitrary time window (contrast with Approach #2). Every chunk's `dominant_category` is exact and unambiguous, because the chunk boundary is defined by coarse category in the first place — a chunk can never contain lines from two different coarse categories.

One distinction from #3 worth keeping in mind: a chunk's *exact speaker* is no longer guaranteed to be a single person, since adjacent turns from different fine-grained speakers sharing a coarse category (e.g. `EYEWITNESS_01` then `EYEWITNESS_02`) now merge into one chunk. The `category` column stays exact; the underlying `speaker`/`speakers` fields on each chunk record which individual(s) actually spoke it, if you want to dig into that later.

In [ ]:
import pandas as pd

category_df = pd.DataFrame({
    'topic_id': topics,
    'category': [m['category'] for m in all_metadata],
})

# Cross-tab: how many chunks of each category landed in each topic
category_crosstab = pd.crosstab(category_df['topic_id'], category_df['category'])
# print(category_crosstab)

# Native BERTopic per-class breakdown
topics_per_class = topic_model.topics_per_class(
    all_chunks, classes=category_df['category'].tolist()
)
fig_per_class = topic_model.visualize_topics_per_class(topics_per_class, top_n_topics=15)
fig_per_class.update_layout(title='Topics per Speaker Category (Turn-Merge-Coarse Chunks)')
fig_per_class.show()

## 5. Outlier Reduction
Reassigns outlier chunks (-1) to the nearest topic by embedding similarity.
Does not refit the model — just updates topic assignments.

Coarse-category merging absorbs most of the very-short-isolated-turn problem that drove outliers up in #3 (a short backchannel line now usually merges into its neighbors' chunk rather than standing alone against the `MIN_WORDS` floor). Expect an outlier rate closer to #1/#2's than #3's, though still likely somewhat higher than the time-window approaches if any short, isolated coarse-category runs survive merging on their own.

In [ ]:
print(f'Outliers before reduction: {topics.count(-1)} '
      f'({topics.count(-1)/len(topics):.1%})')

new_topics = topic_model.reduce_outliers(
    all_chunks,
    topics,
    strategy='embeddings',
    threshold=0.15  # assign every outlier to its nearest topic
)

# Update the model's internal topic assignments
topic_model.update_topics(
    all_chunks,
    topics=new_topics,
    vectorizer_model=vectorizer_model  # re-apply your custom stopwords
)
topics = new_topics

print(f'Outliers after reduction:  {topics.count(-1)} '
      f'({topics.count(-1)/len(topics):.1%})')
print(f'\nTopic counts after outlier reduction:')
topic_info = topic_model.get_topic_info()
print(topic_info[topic_info.Topic != -1][['Topic', 'Count', 'Name']].to_string(index=False))

## 5a. (Optional) Reduce Number of Topics

In [ ]:
TARGET_TOPICS = 30  # Adjust as needed; set to None to skip reduction

if TARGET_TOPICS and len(set(topics)) - 1 > TARGET_TOPICS:
    topic_model.reduce_topics(all_chunks, nr_topics=TARGET_TOPICS)
    topics = topic_model.topics_
    print(f'Reduced to {len(set(topics)) - 1} topics')
else:
    print('No reduction needed')

# Print topic overview
topic_info = topic_model.get_topic_info()
print('\nTop topics:')
print(topic_info[topic_info.Topic != -1][['Topic', 'Count', 'Name']].head(20).to_string(index=False))

## 7. Export Results

In [ ]:
import pandas as pd

def fmt(seconds: float) -> str:
    """Format seconds as MM:SS for display."""
    return f'{int(seconds)//60:02d}:{int(seconds)%60:02d}'

topic_info_lookup = topic_model.get_topic_info().set_index('Topic')['Name'].to_dict()

results_df = pd.DataFrame({
    'documentary': all_doc_names,
    'start_time':  [fmt(m['start']) for m in all_metadata],
    'end_time':    [fmt(m['end'])   for m in all_metadata],
    'speaker':     [m['speaker']  for m in all_metadata],
    'category':    [m['category'] for m in all_metadata],
    'n_lines':     [m['n_lines']  for m in all_metadata],
    'topic_id':    topic_model.topics_,
    'topic_label': [topic_info_lookup.get(t, 'Outlier') if t != -1 else 'Outlier'
                    for t in topic_model.topics_],
    'text_chunk':  all_chunks
})

df_clean = results_df[results_df.topic_id != -1].copy()
df_clean['topic_label'] = df_clean['topic_id'].map(topic_info_lookup)

## LLM Topic Labeling

In [ ]:
import torch
embedding_model.to('cpu')
torch.cuda.empty_cache()

from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained('principled-intelligence/gemma-4-E4B-it-text-only')
model = AutoModelForCausalLM.from_pretrained(
    'principled-intelligence/gemma-4-E4B-it-text-only',
    torch_dtype=torch.bfloat16,
    device_map='auto'
)

In [ ]:
def label_topic(keywords, documents):
    docs_text = '\n'.join([f'- {d[:200]}' for d in documents[:3]])
    messages = [
        {'role': 'system', 'content': 'You are a concise topic labeler. Output only a 3-5 word label, nothing else.'},
        {'role': 'user', 'content': f'Keywords: {", ".join(keywords)}\n\nDocuments:\n{docs_text}\n\nLabel:'}
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors='pt',
        return_dict=True  # force dict output
    ).to('cuda')
    
    input_ids = inputs['input_ids']  # always extract explicitly
    attention_mask = inputs['attention_mask']
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_new_tokens=12,
            do_sample=False
        )
    return tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True).strip()

new_labels = {}
for _, row in topic_model.get_topic_info()[topic_model.get_topic_info().Topic != -1].iterrows():
    tid = row['Topic']
    keywords = [w for w, _ in topic_model.get_topic(tid)[:8]]
    docs = results_df[results_df.topic_id == tid]['text_chunk'].sample(
        min(3, len(results_df[results_df.topic_id == tid])), random_state=42
    ).tolist()
    label = label_topic(keywords, docs)
    new_labels[tid] = label
    print(f'Topic {tid}: {label}')
    torch.cuda.empty_cache()

topic_model.set_topic_labels(new_labels)

In [ ]:
import urllib.request, re

d3_url = "https://cdnjs.cloudflare.com/ajax/libs/d3/7.9.0/d3.min.js"
with urllib.request.urlopen(d3_url) as r:
    d3_js = r.read().decode('utf-8')

# Kill any closing script tag variation
d3_js = re.sub(r'<(/script)', r'<\\\/\1', d3_js, flags=re.IGNORECASE)

# Verify none remain
remaining = re.findall(r'</script', d3_js, re.IGNORECASE)
print(f"Remaining </script> tags in D3: {len(remaining)}")
print(f"D3 length: {len(d3_js):,} chars")

In [ ]:
from export_html import export_documentary_html

export_documentary_html(
    results_df=results_df,
    topic_model=topic_model,
    new_labels=new_labels,
    embedding_model=embedding_model,
    d3_js=d3_js,
    output_dir='./turn_merge_coarse_bertopic',
    strategy_label='Turn-Merge-Coarse, Speaker-Structure',
)